# Self-Attention aplicado a Windows Event Logs (Demo SOC)
Este notebook muestra cómo un **Transformer** (self-attention) puede:
- encontrar relaciones semánticas entre eventos Windows,
- visualizar qué eventos "se miran" entre sí,
- identificar eventos anómalos **sin reglas** (mini-UEBA).

**Importante:** es una demo educativa. Los pesos de atención aquí son aleatorios para ilustrar el mecanismo, pero los embeddings son reales.


In [ ]:
# Instalación de dependencias
!pip -q install sentence-transformers transformers matplotlib


## 1) Dataset de Windows Security Events (ejemplo realista)


In [ ]:
windows_logs = [
    "Event ID 4624: An account was successfully logged on. User: Administrator, Logon Type: 10, Source IP: 192.168.1.15",
    "Event ID 4625: An account failed to log on. User: guest, Logon Type: 3, Source IP: 192.168.1.55",
    "Event ID 4625: Failed logon attempt. User: admin, Logon Type: 10, Source IP: 192.168.1.200",
    "Event ID 4672: Special privileges assigned to new logon. User: Administrator",
    "Event ID 4688: A new process has been created. Process: powershell.exe, Parent: cmd.exe",
    "Event ID 4104: PowerShell script block logging detected. Suspicious command: Invoke-WebRequest",
    "Event ID 5156: The Windows Filtering Platform has permitted a connection. Application: svchost.exe, Dest IP: 192.168.1.20, Port: 443",
    "Event ID 4104: PowerShell script executed with suspicious obfuscation. Command: IEX(New-Object Net.WebClient).DownloadString(...)",
]

print("📌 Eventos cargados:")
for i, l in enumerate(windows_logs):
    print(f"E{i}: {l}")


## 2) Generar embeddings reales (Sentence-Transformers)


In [ ]:
from sentence_transformers import SentenceTransformer
import torch, numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
emb = embed_model.encode(windows_logs, convert_to_tensor=True)

n, d = emb.shape
print("Cantidad de eventos:", n)
print("Dimensión embeddings:", d)


## 3) Self-Attention manual estilo Transformer (Q, K, V)
Usamos la fórmula base:

- Q = X·Wq  
- K = X·Wk  
- V = X·Wv  
- Attention = softmax(QKᵀ/√d)

Para esta demo, Wq, Wk, Wv se inicializan aleatoriamente (ilustración del mecanismo).


In [ ]:
import torch.nn.functional as F

X = emb

torch.manual_seed(42)
Wq = torch.randn(d, d)
Wk = torch.randn(d, d)
Wv = torch.randn(d, d)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

scores = (Q @ K.T) / np.sqrt(d)
attn = F.softmax(scores, dim=1)  # (n, n)

print("Matriz de atención (filas miran, columnas atendidas):")
print(attn.detach().cpu().numpy().round(3))


## 4) Visualizar atención con heatmap (solo matplotlib)


In [ ]:
import matplotlib.pyplot as plt

attn_np = attn.detach().cpu().numpy()

plt.figure(figsize=(8,6))
plt.imshow(attn_np, aspect='auto')
plt.colorbar(label="atención")
plt.xticks(range(n), [f"E{i}" for i in range(n)], rotation=45, ha="right")
plt.yticks(range(n), [f"E{i}" for i in range(n)])
plt.title("Self-Attention entre Windows Event Logs")
plt.xlabel("Evento atendido")
plt.ylabel("Evento que atiende")
plt.tight_layout()
plt.show()


### Cómo leer el heatmap
- **Fila Ei**: qué tan fuerte Ei mira a cada Ej  
- **Columna Ej**: cuánta atención recibe Ej  
- Valores altos ⇒ relación semántica fuerte.

En general:
- 4624/4625 (logons) deberían conectar entre sí.
- 4104/4688 (PowerShell/proceso) deberían conectarse entre sí.
- Un evento raro tiende a quedar con atención baja al grupo o con distribución "extraña".


## 5) Métrica simple de anomalía basada en atención


In [ ]:
def entropy(p):
    p = p + 1e-12
    return -(p * torch.log(p)).sum().item()

entropies = []
mean_attn_to_others = []

for i in range(n):
    row = attn[i]
    entropies.append(entropy(row))
    mean_attn_to_others.append((row.sum() - row[i]).item() / (n-1))

print("ID | Entropía(attn) | Atención media a otros | Evento")
for i, (e, m) in enumerate(zip(entropies, mean_attn_to_others)):
    print(f"E{i:>1} | {e:>13.4f} | {m:>21.4f} | {windows_logs[i]}")


**Interpretación:**
- **Entropía baja:** el evento encuentra pares claros (similaridad fuerte).
- **Entropía alta:** el evento está "confundido" (no encaja bien).
- **Atención media baja:** evento aislado ⇒ posible anomalía.


## 6) Identificar evento más anómalo


In [ ]:
scores_anom = [e + (1 - m) for e, m in zip(entropies, mean_attn_to_others)]
anom_idx = int(np.argmax(scores_anom))

print("Scores de anomalía:")
for i, s in enumerate(scores_anom):
    print(f"E{i}: {s:.4f}")

print("\n🚨 Evento más anómalo según self-attention:")
print(f"E{anom_idx}: {windows_logs[anom_idx]}")


## 7) Prueba con tus eventos reales
Reemplaza `windows_logs` por tus eventos (exportados de SIEM/EDR).  
Tips:
- mezcla eventos normales + 1–2 sospechosos,
- mientras más eventos, mejor contexto.

Si quieres, te lo adapto a:
- Windows Event IDs específicos (Kerberos 4768/4769, 4648, 4697),
- Cortex XDR alerts,
- Sysmon (1, 3, 7, 10, 11, 13),
- Mimikatz / Pass-the-Hash / PSExec.
